# 🎯 Customer Segmentation — Exploratory Data Analysis

**Dataset:** Synthetic Retail Customer Dataset (2000 customers)
**Goal:** Understand customer behavior patterns, feature distributions,
and natural groupings in the data BEFORE applying any ML algorithm.

---
## What This Notebook Explores
1. Dataset structure and data quality
2. Feature distributions and outliers
3. RFM analysis (Recency, Frequency, Monetary)
4. Income vs spending behavior
5. Correlation between features
6. Ground truth archetype validation
7. Feature readiness for K-Means clustering

## 📦 Step 1: Import Libraries

In [ ]:
# pandas: tabular data manipulation — our primary data tool
import pandas as pd

# numpy: mathematical array operations and random number generation
import numpy as np

# matplotlib: core Python plotting library
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec  # For custom subplot layouts

# seaborn: statistical plots with better aesthetics than raw matplotlib
import seaborn as sns

# scipy.stats: statistical functions for distribution analysis
from scipy import stats

# Suppress UserWarnings that are not relevant to our analysis
import warnings
warnings.filterwarnings("ignore")

# Set a uniform visual style for all plots in this notebook
sns.set_theme(style="whitegrid", palette="Set2")

# Default figure size — large enough to read on screen
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"]     = 100

print("All libraries loaded ✅")

## 📂 Step 2: Load Dataset
We load the raw generated customer CSV. At this stage it includes
the archetype labels (A-E) that we injected during generation.
These labels are NOT used by the ML model — they are only here for
us to validate that K-Means recovers similar groupings.

In [ ]:
# Load the customer CSV from the data folder
# This was generated by generate_data.py
df = pd.read_csv("../data/customers.csv")

# Print basic shape to confirm correct load
print(f"Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}")

# Show the data types of every column
print(f"\nData types:\n{df.dtypes}")

# Display first 5 rows to inspect structure visually
df.head()

## 🔍 Step 3: Data Quality Check
Before any analysis, we verify there are no missing values, infinite
values, or data type mismatches. These would silently corrupt our
clustering results later.

In [ ]:
# Check for missing values in every column
# isnull() returns True for NaN values, sum() counts them
print("=== Missing Values ===")
missing = df.isnull().sum()
# Only print columns that have at least one missing value
print(missing[missing > 0] if missing.any() else "No missing values ✅")

# Check for negative values in columns where they are impossible
impossible_negatives = ['annual_income', 'monetary', 'frequency',
                         'spending_score', 'loyalty_years']
print("\n=== Negative Value Check ===")
for col in impossible_negatives:
    # Count values less than 0
    n_neg = (df[col] < 0).sum()
    print(f"  {col}: {n_neg} negative values {'⚠️' if n_neg > 0 else '✅'}")

# Descriptive statistics for all numeric columns
# Shows count, mean, std, min, 25th percentile, median, 75th, max
print("\n=== Statistical Summary ===")
df.describe().round(2)

## 📊 Step 4: Archetype Distribution
We injected 5 customer archetypes during data generation. Let us
verify the distribution before any ML — this is our ground truth.

In [ ]:
# Count how many customers belong to each archetype
archetype_counts = df['archetype'].value_counts().sort_index()

# Full archetype names for display
archetype_labels = {
    'A': 'Premium Loyalists',
    'B': 'Occasional Shoppers',
    'C': 'Bargain Hunters',
    'D': 'At-Risk High-Value',
    'E': 'Young Explorers'
}

# Color each archetype with a distinct color for visual clarity
archetype_colors = ['#F39C12','#3498DB','#2ECC71','#E74C3C','#9B59B6']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Bar chart of customer counts per archetype
bars = axes[0].bar(
    # Map single letter to full name for readable x-axis labels
    [archetype_labels[a] for a in archetype_counts.index],
    archetype_counts.values,
    color=archetype_colors, width=0.6, edgecolor='white'
)
axes[0].set_title('Customer Count by Archetype', fontweight='bold')
axes[0].set_ylabel('Number of Customers')
axes[0].tick_params(axis='x', rotation=20)
# Add count labels on top of each bar
for bar, val in zip(bars, archetype_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 str(val), ha='center', fontweight='bold')

# Plot 2: Pie chart showing proportion of each archetype
axes[1].pie(archetype_counts.values,
            labels=[archetype_labels[a] for a in archetype_counts.index],
            colors=archetype_colors, autopct='%1.1f%%',
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[1].set_title('Archetype Proportion', fontweight='bold')

plt.suptitle('Customer Archetype Distribution (Ground Truth)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 💰 Step 5: RFM Analysis — Recency, Frequency, Monetary
RFM is the gold standard framework for understanding customer value.
- **Recency**: How recently did they buy? (lower = better)
- **Frequency**: How often do they buy? (higher = better)
- **Monetary**: How much do they spend? (higher = better)
Customers who score high on all three are your most valuable.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# --- Row 1: Individual RFM distributions per archetype ---
rfm_features = ['recency_days', 'frequency', 'monetary']
rfm_titles   = ['Recency (Days Since Purchase)', 'Frequency (Purchases/Year)',
                 'Monetary (Annual Spend $)']

for i, (feat, title) in enumerate(zip(rfm_features, rfm_titles)):
    for j, (arch, color) in enumerate(zip(archetype_counts.index, archetype_colors)):
        # Extract values for this archetype only
        subset = df[df['archetype'] == arch][feat]
        # Plot a KDE (smooth density curve) for this archetype's distribution
        # KDE shows the shape of the distribution better than a histogram
        axes[0, i].hist(subset, bins=25, alpha=0.5, color=color,
                         label=archetype_labels[arch], density=True)
    axes[0, i].set_title(f'{title}', fontweight='bold')
    axes[0, i].set_xlabel(feat)
    axes[0, i].set_ylabel('Density')
    if i == 0:
        axes[0, i].legend(fontsize=7)
    # Low recency is GOOD — annotate this to avoid confusion
    if feat == 'recency_days':
        axes[0, i].set_xlabel('Days Since Last Purchase (lower = more recent)')

# --- Row 2: Box plots comparing archetypes on each RFM feature ---
for i, (feat, title) in enumerate(zip(rfm_features, rfm_titles)):
    # Create a list of arrays — one per archetype — for the box plot
    data_per_arch = [df[df['archetype'] == a][feat].values
                     for a in archetype_counts.index]
    # boxplot shows median, IQR, and whiskers (outliers as dots)
    bp = axes[1, i].boxplot(data_per_arch, patch_artist=True,
                             medianprops={'color': 'black', 'linewidth': 2})
    # Color each box to match the archetype color
    for patch, color in zip(bp['boxes'], archetype_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    axes[1, i].set_xticklabels([archetype_labels[a] for a in archetype_counts.index],
                                rotation=15, fontsize=8)
    axes[1, i].set_title(f'{title} — Box Plot', fontweight='bold')

plt.suptitle('RFM Analysis by Customer Archetype', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 💳 Step 6: Income vs Spending Score
The classic segmentation scatter plot. Customers with high income
but low spending score are potentially untapped — they have money
but aren't spending it with us (At-Risk High-Value). Customers
with low income but high spending score are your Bargain Hunters.

In [ ]:
plt.figure(figsize=(10, 7))

# Plot each archetype as a separate scatter layer
for arch, color in zip(archetype_counts.index, archetype_colors):
    subset = df[df['archetype'] == arch]
    # alpha=0.5 makes overlapping points visible
    plt.scatter(subset['annual_income'], subset['spending_score'],
                c=color, label=archetype_labels[arch],
                alpha=0.5, s=20, edgecolors='none')

plt.xlabel('Annual Income ($)', fontsize=12)
plt.ylabel('Spending Score (1-100)', fontsize=12)
plt.title('Annual Income vs Spending Score — Customer Archetypes',
          fontsize=13, fontweight='bold')
plt.legend(loc='upper left', fontsize=9)
# Add quadrant lines at median income and spending score
# These divide the plot into 4 interpretable quadrants
median_income  = df['annual_income'].median()
median_spending = df['spending_score'].median()
plt.axvline(median_income,   color='gray', linestyle='--', alpha=0.5, label='Median income')
plt.axhline(median_spending, color='gray', linestyle='--', alpha=0.5, label='Median spending')
plt.tight_layout()
plt.show()

# Print median values so we understand the quadrant boundaries
print(f"Median annual income:  ${median_income:,.0f}")
print(f"Median spending score: {median_spending:.0f}")

## 🔗 Step 7: Feature Correlation Matrix
Correlation tells us how much features move together.
High correlation between input features can cause K-Means to
effectively double-count similar information.
We check this to validate our feature selection.

In [ ]:
# Select only the numeric ML features for correlation analysis
cluster_features = [
    'annual_income', 'spending_score', 'recency_days', 'frequency',
    'monetary', 'avg_order_value', 'online_purchase_ratio',
    'loyalty_years', 'discount_usage_rate', 'returns_rate',
    'support_tickets', 'clv_score', 'engagement_score'
]

# .corr() computes Pearson correlation between every pair of columns
# Result is a symmetric matrix: corr[A][B] == corr[B][A]
corr_matrix = df[cluster_features].corr()

plt.figure(figsize=(12, 9))

# np.triu creates an upper triangular mask — we only show lower triangle
# to avoid duplicate information (since matrix is symmetric)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

# annot=True shows correlation values inside cells
# cmap='coolwarm': red=positive correlation, blue=negative
# center=0: white = no correlation, extremes are red/blue
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, mask=mask, square=True, linewidths=0.5,
            annot_kws={"size": 8}, cbar_kws={"shrink": 0.8})
plt.title('Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Print highly correlated pairs (abs correlation > 0.7)
print("Highly correlated feature pairs (|r| > 0.7):")
# stack() converts matrix to a Series with MultiIndex (feature1, feature2)
high_corr = corr_matrix.abs().stack()
high_corr = high_corr[(high_corr > 0.7) & (high_corr < 1.0)]
print(high_corr.sort_values(ascending=False).head(10))

## 📊 Step 8: Feature Distribution Histograms
We check each feature's distribution for skewness.
K-Means is sensitive to scale — features with large ranges
dominate the distance calculations. StandardScaler fixes this,
but we need to see the raw distributions first.

In [ ]:
# Create a grid of histograms — one per clustering feature
fig, axes = plt.subplots(3, 5, figsize=(18, 10))
# flatten() turns the 2D grid of axes into a 1D list for easy iteration
axes_flat = axes.flatten()

for i, feat in enumerate(cluster_features):
    # Plot histogram with KDE overlay for this feature
    # bins=30 gives good granularity without over-smoothing
    axes_flat[i].hist(df[feat], bins=30, color='#4A90D9',
                       edgecolor='white', alpha=0.8)
    axes_flat[i].set_title(feat, fontsize=9, fontweight='bold')
    axes_flat[i].set_ylabel('Count', fontsize=8)

    # Compute skewness — positive = right tail, negative = left tail
    # |skewness| > 1 means significantly skewed
    skew = df[feat].skew()
    axes_flat[i].set_xlabel(f'Skew: {skew:.2f}', fontsize=7,
                             color='red' if abs(skew) > 1 else 'gray')

# Hide the unused subplot slots (we have 13 features in 3×5=15 grid)
for j in range(len(cluster_features), len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.suptitle('Feature Distributions — Raw Data Before Scaling',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 📈 Step 9: Customer Lifetime Value Analysis
CLV score is our composite metric for customer value.
We check how it varies by archetype to validate it captures
the business meaning we intended when engineering it.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: CLV score distribution per archetype as violin plot
# Violin plot shows the full distribution shape (not just median/IQR)
# It's like a smooth histogram rotated on its side
clv_data = [df[df['archetype'] == a]['clv_score'].values
            for a in archetype_counts.index]
parts = axes[0].violinplot(clv_data, positions=range(len(archetype_counts)),
                            showmeans=True, showmedians=True)
# Color each violin body to match the archetype color
for j, (body, color) in enumerate(zip(parts['bodies'], archetype_colors)):
    body.set_facecolor(color)
    body.set_alpha(0.7)
axes[0].set_xticks(range(len(archetype_counts)))
axes[0].set_xticklabels([archetype_labels[a] for a in archetype_counts.index],
                         rotation=15, fontsize=9)
axes[0].set_ylabel('CLV Score (proxy)')
axes[0].set_title('CLV Score by Customer Archetype', fontweight='bold')

# Plot 2: Scatter — monetary vs clv_score colored by archetype
for arch, color in zip(archetype_counts.index, archetype_colors):
    sub = df[df['archetype'] == arch]
    axes[1].scatter(sub['monetary'], sub['clv_score'], c=color,
                    alpha=0.4, s=15, label=archetype_labels[arch])
axes[1].set_xlabel('Annual Monetary Spend ($)')
axes[1].set_ylabel('CLV Score')
axes[1].set_title('Monetary Spend vs CLV Score', fontweight='bold')
axes[1].legend(fontsize=8)

plt.suptitle('Customer Lifetime Value Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("Mean CLV Score by Archetype:")
print(df.groupby('archetype')['clv_score'].mean().round(1)
        .map(lambda x: f"{x:,.1f}").rename(archetype_labels))

## 📋 Step 10: Feature Readiness Summary for K-Means

In [ ]:
print("=" * 65)
print("   EDA SUMMARY — FEATURE READINESS FOR K-MEANS")
print("=" * 65)
print(f"  Total customers:          {len(df):,}")
print(f"  Features for clustering:  {len(cluster_features)}")
print(f"  Missing values:           {df[cluster_features].isnull().sum().sum()}")
print()

# Check which features have high skewness (may need log transform)
print("  Skewness check (|skew| > 1 = consider log transform):")
for feat in cluster_features:
    skew = df[feat].skew()
    flag = "⚠️  high skew" if abs(skew) > 1 else "✅"
    print(f"    {feat:<30}: {skew:>6.2f}  {flag}")

print()
print("  Key observations from EDA:")
print("  → Annual income + spending score clearly separate archetypes")
print("  → CLV score captures combined recency+frequency+monetary")
print("  → Discount usage + returns_rate differ strongly by archetype")
print("  → No missing values — data is clean and ready")
print("  → StandardScaler is REQUIRED before K-Means (large value ranges)")
print("  → K=5 is a natural choice — 5 archetypes were injected")
print("=" * 65)